In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score


In [1]:
import pandas as pd

# Load CSV
df = pd.read_csv("../../data/raw/reviews_with_tokens_column.csv")

# Preview the dataset
display(df.head())

,review,language,label,tokens
0,مسحب دجاج و بس,ar,neutral,"['مسحب', 'دجاج', 'و', 'بس']"
1,مطعم لذيذ ومميز في المشويات وخاصه الكباب العرا...,ar,positive,"['مطعم', 'لذيذ', 'ومميز', 'في', 'المشويات', 'و..."
2,طلب سفري واول مرا اطلب والطعم جميل جدا الرز من...,ar,neutral,"['طلب', 'سفري', 'واول', 'مرا', 'اطلب', 'والطعم..."
3,المطعم سيء جدا طلبت مشكل مشاوي محلي وكان لاكل ...,ar,negative,"['المطعم', 'سيء', 'جدا', 'طلبت', 'مشكل', 'مشاو..."
4,مع الاسف كان اسعارهم عاليه وجودتهم ممتازه لكن ...,ar,negative,"['مع', 'الاسف', 'كان', 'اسعارهم', 'عاليه', 'وج..."


In [20]:
import ast

# Convert tokenized string back to list using literal_eval
df['review'] = df['review'].apply(ast.literal_eval)


In [21]:
# Join the tokenized words back into strings
text_data = [" ".join(tokens) for tokens in df['review']]


In [22]:
# Fit TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(text_data)


In [27]:
# Encode labels to numeric values
labels = df['label']
encoder = LabelEncoder()
y = encoder.fit_transform(labels)


In [ ]:
import numpy as np

# Select top 10000 features (adjust as needed)
n_top_features = 10000
feature_sums = np.array(X.sum(axis=0)).flatten()
top_indices = np.argsort(feature_sums)[-n_top_features:]

# Select only top features
X_top = X[:, top_indices]
top_feature_names = np.array(vectorizer.get_feature_names_out())[top_indices]

# Convert to DataFrame and save
tfidf_df = pd.DataFrame(X_top.toarray(), columns=top_feature_names)
tfidf_df['label'] = y
tfidf_df.to_csv("../../data/raw/tfidf_top_features.csv", index=False)
print("Saved CSV with top TF-IDF features.")


Saved CSV with top TF-IDF features.


In [30]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
